## 层归一化

由于梯度消失或梯度爆炸等问题，训练深层神经网络有时会变得具有挑战性。这些问题会导致训练过程不稳定，使网络难以有效地调整权重，从而使学习过程难以找到一组最小化损失函数的参数（权重）​。

层归一化，就是为了提高神经网络训练的稳定性和效率。层归一化的主要思想是调整神经网络层的激活（输出）​，使其均值为0且方差（单位方差）为1。这种调整有助于加速权重的有效收敛，并确保训练过程的一致性和可靠性。


In [42]:
import torch
import torch.nn as nn
torch.manual_seed(123)
batch_example=torch.randn(2,5)
layer=nn.Sequential(nn.Linear(5,6),nn.ReLU())
out=layer(batch_example)
print("out shape:", out.shape)
print(out)

out shape: torch.Size([2, 6])
tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


为了说明问题  我们先进行一些样例实验，如上我们首先实现了一个具有5个输入和6个输出的神经网络层，并将其应用于两个输入示例，我们编写的神经网络层包括一个线性层和一个非线性激活函数ReLU（修正线性单元）​，ReLU是神经网络中的一种标准激活函数

In [43]:
mean=out.mean(dim=-1, keepdim=True)
var=out.var(dim=-1, keepdim=True)
print("mean shape:", mean.shape)
print(mean)
print(var)

mean shape: torch.Size([2, 1])
tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


在对这些输出应用层归一化之前，先检查一下均值和方差,在这里的均值张量中，第1行是第一个输入行的均值，第2行是第二个输入行的均值,在进行均值或方差等运算时，使用keepdim=True可以确保输出张量与输入张量具有相同的维度，如果没有keepdim=True，那么返回的均值张量将是一个二维向量[0.1324, 0.2170]，而不是2×1维的矩阵[​[0.1324],[0.2170]​]。

接下来，对之前得到的层输出进行层归一化操作。具体方法是减去均值，并将结果除以方差的平方根

In [44]:
out_norm=(out-mean)/torch.sqrt(var)
mean=out_norm.mean(dim=-1, keepdim=True)
var=out_norm.var(dim=-1, keepdim=True)
print("out_norm shape:", out_norm.shape)
print(mean)
print(var)

out_norm shape: torch.Size([2, 6])
tensor([[9.9341e-09],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


接下来我们将把这一过程封装成一个PyTorch模块，以便在后续的GPT模型中使用

In [45]:
import torch
import torch.nn as nn
class LayerNorm(nn.Module):
    def __init__(self, emb_dim) -> None:
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))  
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift

这个层归一化的具体实现作用在输入张量x的最后一个维度上，该维度对应于嵌入维度(emb_dim)。变量eps是一个小常数(epsilon)，在归一化过程中会被加到方差上以防止除零错误。

scale和shift是两个可训练的参数（与输入维度相同）​，如果在训练过程中发现调整它们可以改善模型的训练任务表现，那么大语言模型会自动进行调整。这使得模型能够学习适合其数据处理的最佳缩放和偏移。

In [46]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)

ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

# 这里也必须是 keepdim 小写！
mean = out_ln.mean(dim=-1, keepdim=True)
var  = out_ln.var(dim=-1, keepdim=True, unbiased=False)

print("归一化后的均值：\n", mean)
print("归一化后的方差：\n", var)

归一化后的均值：
 tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
归一化后的方差：
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


在我们的方差计算方法中有一个实现细节，即设置unbiased=False。这意味着在方差计算中，我们会使用样本数量n作为方差公式的除数。这种方法没有使用贝塞尔修正。贝塞尔修正通常在样本方差的估计中使用n-1作为分母，以调整偏差。因此，这种方法得到的是所谓有偏方差估计。

对于嵌入维度n非常大的大语言模型（如GPT-2）​，使用n和n-1的差异在实际中几乎可以忽略。我们选择这种方法是为了确保与GPT-2模型的归一化层兼容，并且这种方法反映了TensorFlow的默认行为，因为原始GPT-2模型是用TensorFlow实现的。使用相似的设置可以确保我们的方法与第6章中加载的预训练权重兼容。

与在批次维度上进行归一化的批归一化不同，层归一化是在特征维度上进行归一化。大语言模型通常需要大量的计算资源，训练或推理时的批次大小可能会受到硬件条件或具体用例的影响。由于层归一化是对每个输入独立进行归一化，不受批次大小的限制，因此在这些场景中它提供了更多的灵活性和稳定性。这在分布式训练或在资源受限的环境中部署模型时尤为重要。

## GELU前馈神经网络
![geluForward](imgs/geluForward.png)

ReLU（右）是一个分段线性函数，当输入为正数时直接输出输入值，否则输出0。GELU（左）则是一个平滑的非线性函数，它近似ReLU，但在几乎所有负值（除了在约等于-0.75的位置外）上都有非零梯度

GELU的平滑特性可以在训练过程中带来更好的优化效果，因为它允许模型参数进行更细微的调整。相比之下，ReLU在零点处有一个尖锐的拐角​，有时会使得优化过程更加困难，特别是在深度或复杂的网络结构中。此外，ReLU对负输入的输出为0，而GELU对负输入会输出一个小的非零值。这意味着在训练过程中，接收到负输入的神经元仍然可以参与学习，只是贡献程度不如正输入大。

In [47]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

接下里，我们将使用GELU函数来实现小型神经网络模块FeedForward，该模块将在大语言模型的Transformer块中使用。

In [48]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

pip install torchsummary

In [49]:
from torchsummary import summary
class FeedForward(nn.Module):
    def __init__(self, cfg) -> None:
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"])
        )
    
    def forward(self,x):
        return self.layers(x)

ffn=FeedForward(GPT_CONFIG_124M)
summary(ffn,input_size=(2,3,768))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1           [-1, 2, 3, 3072]       2,362,368
              GELU-2           [-1, 2, 3, 3072]               0
            Linear-3            [-1, 2, 3, 768]       2,360,064
Total params: 4,722,432
Trainable params: 4,722,432
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.02
Forward/backward pass size (MB): 0.32
Params size (MB): 18.01
Estimated Total Size (MB): 18.35
----------------------------------------------------------------


该模块的输入和输出维度保持一致，但它通过第一个线性层将嵌入维度扩展到了更高的维度。

扩展之后，应用非线性GELU激活函数，然后通过第二个线性变换将维度缩回原始大小。这种设计允许模型探索更丰富的表示空间

此外，输入维度和输出维度的一致性简化了架构，使我们在后续堆叠多个层时无须调整维度，从而增强了模型的扩展能力。

## 添加残差连接

残差连接最初用于计算机视觉中的深度网络（特别是残差网络）​，目的是缓解梯度消失问题。梯度消失问题指的是在训练过程中，梯度在反向传播时逐渐变小，导致早期网络层难以有效训练。

![shortcutForward](imgs/shortcutForward.png)

In [50]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self,layer_sizes,use_shortcut) -> None:
        super().__init__()
        self.layers=nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0],layer_sizes[1]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1],layer_sizes[2]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2],layer_sizes[3]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3],layer_sizes[4]),GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4],layer_sizes[5]),GELU()),
        ])
        self.use_shortcut=use_shortcut
    def forward(self,x):
        for layer in self.layers:
            layer_output=layer(x)
            if self.use_shortcut and x.shape==layer_output.shape:
                x=x+layer_output
            else:
                x=layer_output
        return x

上述代码实现了一个具有5层的深度神经网络，每层由一个线性层和一个GELU激活函数组成。在前向传播过程中，我们通过各层迭代地传递输入。并且，如果self.use_shortcut属性被设置为True，那么我们就会选择性地添加图4-12中所示的快捷连接

我们先使用这段代码初始化一个没有快捷连接的神经网络。每一层将被初始化为接受包含3个输入值的样本，并返回3个输出值。最后一层会返回一个输出值

In [ ]:
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])

    loss = nn.MSELoss()
    loss = loss(output, target)
    
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")


这段代码定义了一个损失函数，用于计算模型输出与用户指定目标（为简化处理，这里设为0）的接近程度。然后，当调用loss.backward()时，PyTorch会计算模型中每一层的损失梯度。我们通过model.named_parameters()迭代权重参数。假设某一层有一个3×3的权重参数矩阵，那么该层将有3×3的梯度值。我们打印这3×3的梯度值的平均绝对值，以得到每一层的单一梯度值，从而更容易比较层与层之间的梯度。

接下来，使用print_gradients函数，并将其应用于没有快捷连接的模型

In [51]:
layer_sizes = [3, 3, 3, 3, 3, 1]  
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)


print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173584925942123
layers.1.0.weight has gradient mean of 0.00012011159560643137
layers.2.0.weight has gradient mean of 0.0007152040489017963
layers.3.0.weight has gradient mean of 0.0013988736318424344
layers.4.0.weight has gradient mean of 0.005049645435065031


print_gradients函数的输出显示，梯度在从最后一层(layers.4)到第1层(layers.0)的过程中逐渐变小，这种现象称为梯度消失问题。

现在，实例化一个包含残差连接的模型，并观察它的比较结果

In [52]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694105327129364
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258540630340576


最后一层(layers.4)的梯度仍然大于其他层。然而，梯度值在逐渐接近第1层(layers.0)时趋于稳定，并且没有缩小到几乎消失的程度。总之，快捷连接在解决深度神经网络中梯度消失问题的限制方面非常重要。快捷连接是诸如大语言模型等超大规模模型的核心构建块，它们将通过确保各层之间梯度的稳定流动来帮助实现更有效的训练。

## 构建transforer模块

![transformerBlock](imgs/transformerBlock.png)

一个Transformer块结合了多个组件，包括掩码多头注意力模块和我们之前实现的FeedForward模块​。

当Transformer块处理输入序列时，序列中的每个元素（如单词或子词）都被表示为一个固定大小的向量（此处为768维）​。Transformer块内的操作，包括多头注意力和前馈层，旨在以保持这些向量维度的方式来转换它们。

Transformer块的核心思想是，自注意力机制在多头注意力块中用于识别和分析输入序列中元素之间的关系。相比之下，前馈神经网络则在每个位置上对数据进行单独的修改。这种组合不仅提供了对输入更细致的理解和处理，而且提升了模型处理复杂数据模式的整体能力。